# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_03 — Volatility Targeting and Position Sizing

### Research question

How do we convert alpha signals and return forecasts into position sizes while controlling volatility, leverage, turnover, and drawdown risk?

This notebook follows:

```text
04_01_multi_asset_return_engine
04_02_capm_and_performance_attribution
```

The previous notebook explained where portfolio returns came from. This notebook asks:

> Given a set of returns or signals, how large should each position be?

It covers:

1. realised volatility estimation;
2. rolling volatility;
3. EWMA volatility;
4. target-volatility scaling;
5. leverage caps;
6. inverse-volatility sizing;
7. signal-strength sizing;
8. portfolio-level volatility targeting;
9. asset-level risk budgets;
10. transaction costs and turnover;
11. volatility-targeted NAV;
12. drawdown-aware scaling;
13. stress-regime behaviour;
14. procyclicality warnings;
15. futures-style contract sizing;
16. audit checks;
17. saved outputs and manifest.

The key idea:

> Position sizing is where forecasts become risk. A weak signal with good sizing can survive; a decent signal with reckless sizing can blow up.

## 1. Why volatility targeting matters

Many strategies have time-varying risk.

A fixed notional position may be:

- too small during calm periods;
- too large during volatile periods;
- catastrophic during volatility spikes.

Volatility targeting scales exposure to target a desired annualised volatility:

$$
scale_t = \frac{\sigma_{\text{target}}}{\hat\sigma_t}
$$

If forecast volatility rises, exposure falls.

If forecast volatility falls, exposure rises.

But volatility targeting is not magic:

1. volatility forecasts lag;
2. leverage can rise in calm markets;
3. transaction costs increase;
4. deleveraging can occur after losses;
5. crowding can make vol-targeting procyclical.

This notebook treats vol targeting as a risk-control technique, not a free Sharpe booster.

## 2. Position sizing formulas

### Single-asset target-vol position

For a signal $s_t$ and forecast volatility $\hat\sigma_t$:

$$
w_t = s_t \frac{\sigma_{\text{target}}}{\hat\sigma_t}
$$

Then cap leverage:

$$
w_t^{cap} = clip(w_t,-L,L)
$$

### Multi-asset inverse-vol weight

$$
w_{i,t} = \frac{1/\hat\sigma_{i,t}} {\sum_j 1/\hat\sigma_{j,t}}
$$

### Risk-budget weight

For risk budgets $b_i$:

$$
w_{i,t} = \frac{b_i/\hat\sigma_{i,t}} {\sum_j b_j/\hat\sigma_{j,t}}
$$

### Portfolio-level volatility targeting

If a raw portfolio has forecast volatility $\hat\sigma^p_t$:

$$
w^{target}_{i,t} = w^{raw}_{i,t} \frac{\sigma^p_{\text{target}}}{\hat\sigma^p_t}
$$

The timing matters:

> volatility estimated through $t$ should size positions for $t+1$.

## 3. Transaction costs and turnover

Turnover is:

$$
TO_t = \sum_i |w_{i,t} - w_{i,t-1}|
$$

Transaction-cost drag:

$$
cost_t = \sum_i |w_{i,t} - w_{i,t-1}|c_i
$$

A volatility targeting rule can look excellent before costs and mediocre after costs.

This notebook reports both.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class VolTargetConfig:
    n_days: int = 2_000
    n_assets: int = 8
    annualisation: int = 252
    rolling_vol_window: int = 63
    ewma_lambda: float = 0.94
    target_portfolio_vol: float = 0.12
    target_asset_vol: float = 0.15
    max_asset_leverage: float = 2.0
    max_portfolio_gross: float = 3.0
    transaction_cost_bps: float = 2.0
    drawdown_delever_threshold: float = -0.10
    drawdown_delever_multiplier: float = 0.50
    initial_capital: float = 1_000_000.0
    seed: int = 42


config = VolTargetConfig()
config

## 5. Simulate multi-asset returns

We simulate a diversified universe:

- equities;
- bonds;
- commodities;
- FX;
- crypto-like high-vol asset.

The simulation includes:

1. correlated shocks;
2. regime-switching volatility;
3. occasional stress events;
4. weak trend signals;
5. asset-specific volatilities.

This gives the sizing rules something realistic to react to.

In [ ]:
def simulate_multi_asset_returns(config: VolTargetConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2016-01-01", periods=config.n_days)

    assets = [
        "US_EQ",
        "EU_EQ",
        "US_BOND",
        "EU_BOND",
        "GOLD",
        "OIL",
        "FX_CARRY",
        "BTC_PROXY",
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "FX_CARRY": "fx",
        "BTC_PROXY": "crypto",
    }

    annual_vols = np.array([0.16, 0.18, 0.06, 0.07, 0.18, 0.32, 0.10, 0.65])
    annual_means = np.array([0.07, 0.06, 0.025, 0.02, 0.035, 0.04, 0.03, 0.12])

    daily_vols = annual_vols / np.sqrt(config.annualisation)
    daily_means = annual_means / config.annualisation

    corr = np.array([
        [ 1.00,  0.75, -0.25, -0.20,  0.05,  0.25,  0.10,  0.35],
        [ 0.75,  1.00, -0.20, -0.25,  0.05,  0.20,  0.15,  0.30],
        [-0.25, -0.20,  1.00,  0.70,  0.15, -0.10, -0.05, -0.20],
        [-0.20, -0.25,  0.70,  1.00,  0.10, -0.10, -0.05, -0.15],
        [ 0.05,  0.05,  0.15,  0.10,  1.00,  0.20,  0.05,  0.15],
        [ 0.25,  0.20, -0.10, -0.10,  0.20,  1.00,  0.10,  0.25],
        [ 0.10,  0.15, -0.05, -0.05,  0.05,  0.10,  1.00,  0.10],
        [ 0.35,  0.30, -0.20, -0.15,  0.15,  0.25,  0.10,  1.00],
    ])

    base_cov = np.outer(daily_vols, daily_vols) * corr

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.045, 0.955]])

    returns = np.zeros((config.n_days, len(assets)))
    latent_trend = np.zeros((config.n_days, len(assets)))

    for t in range(1, config.n_days):
        regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.3
        cov_t = base_cov * vol_multiplier**2

        shock = rng.multivariate_normal(daily_means, cov_t)

        # Stress days.
        if rng.uniform() < 0.008:
            shock[0] -= rng.uniform(0.025, 0.070)
            shock[1] -= rng.uniform(0.025, 0.070)
            shock[2] += rng.uniform(0.003, 0.020)
            shock[3] += rng.uniform(0.003, 0.020)
            shock[7] -= rng.uniform(0.050, 0.150)

        latent_trend[t] = 0.97 * latent_trend[t - 1] + 0.15 * shock + 0.05 * rng.standard_normal(len(assets))
        trend_alpha = 0.08 * latent_trend[t - 1] * daily_vols

        returns[t] = shock + trend_alpha

    returns_df = pd.DataFrame(returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "annual_vol_assumption": annual_vols,
        "annual_mean_assumption": annual_means,
        "cost_bps": config.transaction_cost_bps,
        "contract_multiplier": [1, 1, 1, 1, 100, 1000, 1, 1],
        "is_futures": [False, False, False, False, True, True, False, False]
    })

    return returns_df, metadata


returns_df, asset_metadata = simulate_multi_asset_returns(config)

returns_df.head(), asset_metadata

In [ ]:
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.85)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(returns_df["date"], returns_df["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Return diagnostics

Before sizing positions, inspect realised return properties.

This catches scale problems and tells us which assets require smaller notional allocations.

In [ ]:
return_summary = returns.agg(["mean", "std", "min", "max"]).T
return_summary = return_summary.rename(columns={
    "mean": "daily_mean",
    "std": "daily_vol",
    "min": "min_daily_return",
    "max": "max_daily_return",
})
return_summary["annualised_mean"] = return_summary["daily_mean"] * config.annualisation
return_summary["annualised_vol"] = return_summary["daily_vol"] * np.sqrt(config.annualisation)
return_summary["asset_class"] = return_summary.index.map(asset_metadata.set_index("asset")["asset_class"])

return_summary.sort_values("annualised_vol", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
vol_plot = return_summary.sort_values("annualised_vol")
plt.barh(vol_plot.index, vol_plot["annualised_vol"])
plt.title("Realised Annualised Volatility")
plt.xlabel("Annualised volatility")
plt.ylabel("Asset")
plt.show()

## 7. Volatility estimators

We implement two simple estimators.

### Rolling volatility

$$
\hat\sigma_{t}^{roll} = \sqrt{252} \operatorname{std}(r_{t-w+1},\dots,r_t)
$$

### EWMA volatility

$$
\begin{aligned}
\hat\sigma_t^2 &= \lambda \hat\sigma_{t-1}^2 \\
&\quad + (1-\lambda)r_t^2
\end{aligned}
$$

Both estimates are shifted by one day before sizing positions:

$$
w_t \text{ uses } \hat\sigma_{t-1}
$$

This avoids look-ahead.

In [ ]:
def rolling_volatility(returns: pd.DataFrame, window: int, annualisation: int) -> pd.DataFrame:
    return returns.rolling(window, min_periods=max(20, window // 3)).std() * np.sqrt(annualisation)


def ewma_volatility(returns: pd.DataFrame, lam: float, annualisation: int) -> pd.DataFrame:
    var = pd.DataFrame(index=returns.index, columns=returns.columns, dtype=float)

    initial_var = returns.iloc[:30].var().fillna(returns.var())
    var.iloc[0] = initial_var

    for i in range(1, len(returns)):
        var.iloc[i] = lam * var.iloc[i - 1] + (1 - lam) * returns.iloc[i - 1] ** 2

    return np.sqrt(var) * np.sqrt(annualisation)


roll_vol_raw = rolling_volatility(returns, config.rolling_vol_window, config.annualisation)
ewma_vol_raw = ewma_volatility(returns, config.ewma_lambda, config.annualisation)

# Shift so today's position uses yesterday's volatility estimate.
roll_vol = roll_vol_raw.shift(1)
ewma_vol = ewma_vol_raw.shift(1)

# Fill early estimates with expanding volatility.
expanding_vol = returns.expanding(min_periods=20).std() * np.sqrt(config.annualisation)
roll_vol = roll_vol.fillna(expanding_vol.shift(1)).fillna(return_summary["annualised_vol"])
ewma_vol = ewma_vol.fillna(expanding_vol.shift(1)).fillna(return_summary["annualised_vol"])

roll_vol.head()

In [ ]:
plt.figure(figsize=(12, 6))
for asset in ["US_EQ", "US_BOND", "OIL", "BTC_PROXY"]:
    plt.plot(ewma_vol.index, ewma_vol[asset], label=f"{asset} EWMA")
plt.title("EWMA Volatility Estimates")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 8. Volatility forecast diagnostics

A volatility estimator should react to realised volatility but not use future returns.

We compare estimated volatility to future 21-day realised volatility.

This is diagnostic only, not a full volatility forecasting study.

In [ ]:
future_realised_vol_21 = (
    returns
    .rolling(21)
    .std()
    .shift(-21)
    * np.sqrt(config.annualisation)
)

vol_forecast_diagnostics = []

for asset in asset_cols:
    joined = pd.DataFrame({
        "ewma_vol": ewma_vol[asset],
        "rolling_vol": roll_vol[asset],
        "future_rv_21": future_realised_vol_21[asset]
    }).dropna()

    vol_forecast_diagnostics.append({
        "asset": asset,
        "corr_ewma_future_rv": joined["ewma_vol"].corr(joined["future_rv_21"]),
        "corr_rolling_future_rv": joined["rolling_vol"].corr(joined["future_rv_21"]),
        "mean_ewma_vol": joined["ewma_vol"].mean(),
        "mean_future_rv": joined["future_rv_21"].mean()
    })

vol_forecast_diagnostics = pd.DataFrame(vol_forecast_diagnostics)
vol_forecast_diagnostics

## 9. Alpha signal generation

We create simple synthetic signals:

1. 63-day trend signal;
2. 5-day reversal signal;
3. volatility carry penalty;
4. asset-class risk overlay.

Signal values are clipped to $[-1,1]$.

The point is not to build a great alpha model. The point is to test position sizing.

In [ ]:
def generate_signals(returns: pd.DataFrame) -> pd.DataFrame:
    trend_63 = (1 + returns).rolling(63, min_periods=20).apply(lambda x: np.prod(x) - 1, raw=True)
    reversal_5 = -(1 + returns).rolling(5, min_periods=5).apply(lambda x: np.prod(x) - 1, raw=True)
    vol_penalty = -returns.rolling(21, min_periods=10).std()

    # Cross-sectional ranks by date.
    trend_rank = trend_63.rank(axis=1, pct=True) - 0.5
    reversal_rank = reversal_5.rank(axis=1, pct=True) - 0.5
    vol_rank = vol_penalty.rank(axis=1, pct=True) - 0.5

    signal = 0.60 * trend_rank + 0.25 * reversal_rank + 0.15 * vol_rank
    signal = 2.0 * signal
    signal = signal.clip(-1.0, 1.0)

    return signal.shift(1).fillna(0.0)


signals = generate_signals(returns)
signals.head()

In [ ]:
plt.figure(figsize=(12, 6))
for asset in ["US_EQ", "US_BOND", "GOLD", "BTC_PROXY"]:
    plt.plot(signals.index, signals[asset], label=asset, alpha=0.85)
plt.axhline(0, linestyle="--")
plt.title("Synthetic Signals")
plt.xlabel("Date")
plt.ylabel("Signal")
plt.legend()
plt.show()

## 10. Baseline equal-weight portfolio

The simplest benchmark is equal weight:

$$
w_{i,t}=\frac{1}{N}
$$

This is our baseline for comparing sizing rules.

In [ ]:
def equal_weight_positions(returns: pd.DataFrame) -> pd.DataFrame:
    n = returns.shape[1]
    return pd.DataFrame(1.0 / n, index=returns.index, columns=returns.columns)


weights_equal = equal_weight_positions(returns)

weights_equal.head()

## 11. Inverse-volatility sizing

Inverse-vol weights:

$$
w_{i,t} = \frac{1/\hat\sigma_{i,t}} {\sum_j 1/\hat\sigma_{j,t}}
$$

This gives less weight to volatile assets.

We use EWMA volatility estimates shifted by one day.

In [ ]:
def inverse_vol_weights(vol_forecast: pd.DataFrame) -> pd.DataFrame:
    inv = 1.0 / vol_forecast.replace(0, np.nan)
    inv = inv.replace([np.inf, -np.inf], np.nan)
    weights = inv.div(inv.sum(axis=1), axis=0)
    return weights.fillna(0.0)


weights_inverse_vol = inverse_vol_weights(ewma_vol)

weights_inverse_vol.head()

## 12. Asset-level volatility-targeted signal sizing

Signal-driven position:

$$
w_{i,t} = s_{i,t} \frac{\sigma_{\text{asset target}}}{\hat\sigma_{i,t}}
$$

Then cap:

$$
w_{i,t}=clip(w_{i,t},-L,L)
$$

Finally, if gross exposure exceeds a portfolio cap, scale all positions down.

In [ ]:
def cap_gross_exposure(weights: pd.DataFrame, max_gross: float) -> pd.DataFrame:
    gross = weights.abs().sum(axis=1)
    scale = np.minimum(1.0, max_gross / gross.replace(0, np.nan)).fillna(1.0)
    return weights.multiply(scale, axis=0)


def signal_vol_target_weights(signals, vol_forecast, target_asset_vol, max_asset_leverage, max_portfolio_gross):
    raw = signals * (target_asset_vol / vol_forecast.replace(0, np.nan))
    raw = raw.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    capped = raw.clip(lower=-max_asset_leverage, upper=max_asset_leverage)
    capped = cap_gross_exposure(capped, max_portfolio_gross)
    return capped


weights_signal_asset_vol = signal_vol_target_weights(
    signals,
    ewma_vol,
    config.target_asset_vol,
    config.max_asset_leverage,
    config.max_portfolio_gross
)

weights_signal_asset_vol.head()

## 13. Portfolio-level volatility targeting

First construct raw signal weights.

Then estimate the portfolio's predicted volatility:

$$
\hat\sigma^p_t = \sqrt{ w_t^\top \hat\Sigma_t w_t }
$$

This notebook uses a diagonal covariance approximation:

$$
\hat\sigma^p_t \approx \sqrt{\sum_i (w_{i,t}\hat\sigma_{i,t})^2}
$$

Then scale:

$$
w^{target}_t = w^{raw}_t \frac{\sigma^p_{target}}{\hat\sigma^p_t}
$$

and cap gross exposure.

In [ ]:
def portfolio_vol_estimate_diagonal(weights: pd.DataFrame, vol_forecast: pd.DataFrame) -> pd.Series:
    daily_asset_vol = vol_forecast / np.sqrt(config.annualisation)
    daily_port_var = ((weights * daily_asset_vol) ** 2).sum(axis=1)
    return np.sqrt(daily_port_var) * np.sqrt(config.annualisation)


def portfolio_vol_target_weights(raw_weights, vol_forecast, target_portfolio_vol, max_portfolio_gross):
    forecast_port_vol = portfolio_vol_estimate_diagonal(raw_weights, vol_forecast)
    scale = target_portfolio_vol / forecast_port_vol.replace(0, np.nan)
    scaled = raw_weights.multiply(scale, axis=0)
    scaled = scaled.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaled = cap_gross_exposure(scaled, max_portfolio_gross)
    return scaled, forecast_port_vol, scale


raw_signal_weights = signals.div(signals.abs().sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
raw_signal_weights = raw_signal_weights * 1.0

weights_signal_port_vol, raw_signal_port_vol_forecast, port_vol_scale = portfolio_vol_target_weights(
    raw_signal_weights,
    ewma_vol,
    config.target_portfolio_vol,
    config.max_portfolio_gross
)

weights_signal_port_vol.head()

## 14. Risk-budget sizing

Risk budgets allocate intended risk shares.

For example:

- equities: 30%;
- bonds: 20%;
- commodities: 25%;
- FX: 10%;
- crypto: 15%.

Within assets, use inverse volatility.

Risk-budget weights:

$$
w_{i,t} = \frac{b_i/\hat\sigma_{i,t}} {\sum_j b_j/\hat\sigma_{j,t}}
$$

In [ ]:
def risk_budget_weights(vol_forecast, asset_metadata):
    asset_class_budget = {
        "equity": 0.30,
        "bond": 0.20,
        "commodity": 0.25,
        "fx": 0.10,
        "crypto": 0.15,
    }

    class_map = asset_metadata.set_index("asset")["asset_class"].to_dict()
    assets = vol_forecast.columns

    asset_budget = pd.Series(0.0, index=assets)
    for asset_class, budget in asset_class_budget.items():
        class_assets = [a for a in assets if class_map[a] == asset_class]
        if class_assets:
            asset_budget.loc[class_assets] = budget / len(class_assets)

    raw = asset_budget / vol_forecast.replace(0, np.nan)
    weights = raw.div(raw.sum(axis=1), axis=0)
    return weights.fillna(0.0)


weights_risk_budget = risk_budget_weights(ewma_vol, asset_metadata)

weights_risk_budget.head()

## 15. Backtest engine for sizing rules

Weights at time $t-1$ are applied to return at time $t$:

$$
R^p_t = \sum_i w_{i,t-1}R_{i,t} - cost_t
$$

Costs:

$$
cost_t = c\sum_i |w_{i,t}-w_{i,t-1}|
$$

This is intentionally simple but strict about timing.

In [ ]:
def backtest_weights(weights, returns, cost_bps, initial_capital):
    # Position decided today is applied to tomorrow's return.
    investable = weights.shift(1).fillna(0.0)

    gross_return = (investable * returns).sum(axis=1)

    trade = weights.diff().abs().fillna(weights.abs())
    turnover = trade.sum(axis=1)
    cost = turnover * (cost_bps / 10000.0)

    net_return = gross_return - cost
    nav = initial_capital * (1 + net_return).cumprod()

    out = pd.DataFrame({
        "gross_return": gross_return,
        "transaction_cost": cost,
        "net_return": net_return,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": investable.abs().sum(axis=1),
        "net_exposure": investable.sum(axis=1),
    })

    asset_contrib = investable * returns
    return out, asset_contrib, investable


strategies = {
    "equal_weight": weights_equal,
    "inverse_vol": weights_inverse_vol,
    "risk_budget": weights_risk_budget,
    "signal_asset_vol_target": weights_signal_asset_vol,
    "signal_portfolio_vol_target": weights_signal_port_vol,
}

backtests = {}
contributions = {}
investable_weights = {}

for name, w in strategies.items():
    bt, contrib, inv_w = backtest_weights(
        w,
        returns,
        config.transaction_cost_bps,
        config.initial_capital
    )
    backtests[name] = bt
    contributions[name] = contrib
    investable_weights[name] = inv_w

list(backtests.keys())

## 16. Performance summary

For each sizing rule, compute:

- annualised return;
- annualised volatility;
- Sharpe proxy;
- max drawdown;
- average turnover;
- average cost;
- average gross exposure.

In [ ]:
def max_drawdown(nav):
    running_max = nav.cummax()
    dd = nav / running_max - 1.0
    return dd, float(dd.min())


def performance_summary(backtests, config):
    rows = []

    for name, bt in backtests.items():
        r = bt["net_return"]
        dd, mdd = max_drawdown(bt["nav"])

        rows.append({
            "strategy": name,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "max_drawdown": mdd,
            "total_return": float(bt["nav"].iloc[-1] / bt["nav"].iloc[0] - 1.0),
            "mean_turnover": float(bt["turnover"].mean()),
            "annualised_cost_drag": float(bt["transaction_cost"].mean() * config.annualisation),
            "mean_gross_exposure": float(bt["gross_exposure"].mean()),
            "p95_gross_exposure": float(bt["gross_exposure"].quantile(0.95)),
        })

    return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


perf = performance_summary(backtests, config)
perf

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in backtests.items():
    plt.plot(bt.index, bt["nav"] / bt["nav"].iloc[0], label=name)
plt.title("Sizing Strategy NAV Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.barh(perf.sort_values("annualised_vol")["strategy"], perf.sort_values("annualised_vol")["annualised_vol"])
plt.title("Realised Annualised Volatility by Strategy")
plt.xlabel("Annualised volatility")
plt.ylabel("Strategy")
plt.show()

## 17. Realised versus target volatility

Volatility-targeted strategies should be compared with their intended target.

We compute rolling realised strategy volatility:

$$
\sigma^p_{realised,t} = \sqrt{252} \operatorname{std}(R^p_{t-w+1},\dots,R^p_t)
$$

In [ ]:
strategy_rolling_vol = pd.DataFrame(index=returns.index)

for name, bt in backtests.items():
    strategy_rolling_vol[name] = bt["net_return"].rolling(63, min_periods=20).std() * np.sqrt(config.annualisation)

strategy_rolling_vol.tail()

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["equal_weight", "inverse_vol", "signal_asset_vol_target", "signal_portfolio_vol_target"]:
    plt.plot(strategy_rolling_vol.index, strategy_rolling_vol[name], label=name)
plt.axhline(config.target_portfolio_vol, linestyle="--", label="portfolio target vol")
plt.title("Rolling Realised Portfolio Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 18. Exposure behaviour

Vol-targeting changes exposure through time.

High exposure in calm periods and low exposure in volatile periods is expected.

The risk is that exposure peaks just before a volatility shock.

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["equal_weight", "inverse_vol", "risk_budget", "signal_asset_vol_target", "signal_portfolio_vol_target"]:
    plt.plot(backtests[name].index, backtests[name]["gross_exposure"], label=name)
plt.axhline(config.max_portfolio_gross, linestyle="--", label="gross cap")
plt.title("Gross Exposure Through Time")
plt.xlabel("Date")
plt.ylabel("Gross exposure")
plt.legend()
plt.show()

## 19. Drawdown-aware deleveraging overlay

A drawdown-aware overlay reduces exposure when the strategy is in drawdown.

Rule:

$$
scale_t =
\begin{cases}
1, & DD_t > threshold \\
m, & DD_t \le threshold
\end{cases}
$$

This is simple and not necessarily optimal.

It can reduce tail risk, but it can also lock in losses and miss recoveries.

In [ ]:
def apply_drawdown_overlay(weights, returns, base_backtest, threshold, multiplier):
    dd, _ = max_drawdown(base_backtest["nav"])
    scale = pd.Series(1.0, index=weights.index)
    scale.loc[dd <= threshold] = multiplier

    # Use yesterday's drawdown state to scale today's desired weights.
    scaled_weights = weights.multiply(scale.shift(1).fillna(1.0), axis=0)
    return scaled_weights, scale


weights_signal_port_vol_dd, drawdown_scale = apply_drawdown_overlay(
    weights_signal_port_vol,
    returns,
    backtests["signal_portfolio_vol_target"],
    config.drawdown_delever_threshold,
    config.drawdown_delever_multiplier
)

bt_dd, contrib_dd, inv_w_dd = backtest_weights(
    weights_signal_port_vol_dd,
    returns,
    config.transaction_cost_bps,
    config.initial_capital
)

backtests["signal_portfolio_vol_target_drawdown_overlay"] = bt_dd
contributions["signal_portfolio_vol_target_drawdown_overlay"] = contrib_dd
investable_weights["signal_portfolio_vol_target_drawdown_overlay"] = inv_w_dd

perf_with_overlay = performance_summary(backtests, config)
perf_with_overlay

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(backtests["signal_portfolio_vol_target"].index, backtests["signal_portfolio_vol_target"]["nav"], label="vol target")
plt.plot(bt_dd.index, bt_dd["nav"], label="vol target + drawdown overlay")
plt.title("Drawdown Overlay Comparison")
plt.xlabel("Date")
plt.ylabel("NAV")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(drawdown_scale.index, drawdown_scale)
plt.title("Drawdown Overlay Scale")
plt.xlabel("Date")
plt.ylabel("Scale")
plt.show()

## 20. Stress-regime analysis

Volatility targeting often looks good until stress arrives.

We compare performance on calm and high-volatility regimes.

In [ ]:
regime_by_date = returns_df.set_index("date")["regime"].reindex(returns.index)

regime_rows = []

for name, bt in backtests.items():
    df = bt.copy()
    df["regime"] = regime_by_date

    for regime_value, g in df.groupby("regime"):
        regime_rows.append({
            "strategy": name,
            "regime": int(regime_value),
            "n_days": len(g),
            "mean_return_ann": float(g["net_return"].mean() * config.annualisation),
            "vol_ann": float(g["net_return"].std(ddof=1) * np.sqrt(config.annualisation)),
            "worst_day": float(g["net_return"].min()),
            "mean_gross_exposure": float(g["gross_exposure"].mean()),
            "mean_turnover": float(g["turnover"].mean()),
        })

regime_analysis = pd.DataFrame(regime_rows)
regime_analysis.head()

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["equal_weight", "signal_portfolio_vol_target", "signal_portfolio_vol_target_drawdown_overlay"]:
    sub = regime_analysis[regime_analysis["strategy"] == name]
    plt.plot(sub["regime"], sub["vol_ann"], marker="o", label=name)
plt.title("Realised Volatility by Regime")
plt.xlabel("Regime")
plt.ylabel("Annualised volatility")
plt.xticks([0, 1])
plt.legend()
plt.show()

## 21. Turnover and cost diagnostics

Position sizing rules can improve volatility but increase turnover.

We inspect turnover and cost drag through time.

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["equal_weight", "inverse_vol", "signal_asset_vol_target", "signal_portfolio_vol_target"]:
    bt = backtests[name]
    plt.plot(bt.index, bt["turnover"].rolling(21).mean(), label=name)
plt.title("21-Day Average Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")
plt.legend()
plt.show()

cost_summary = pd.DataFrame([
    {
        "strategy": name,
        "total_cost_return": float(bt["transaction_cost"].sum()),
        "annualised_cost_drag": float(bt["transaction_cost"].mean() * config.annualisation),
        "mean_turnover": float(bt["turnover"].mean())
    }
    for name, bt in backtests.items()
]).sort_values("annualised_cost_drag", ascending=False)

cost_summary

## 22. Asset-level risk contribution approximation

For a diagonal covariance approximation, asset risk contribution is:

$$
RC_i = \frac{(w_i\sigma_i)^2} {\sum_j(w_j\sigma_j)^2}
$$

This ignores correlations, but it is a useful first diagnostic.

Later notebooks will use full covariance matrices.

In [ ]:
def diagonal_risk_contributions(weights, vol_forecast):
    daily_vol = vol_forecast / np.sqrt(config.annualisation)
    risk_units = (weights * daily_vol) ** 2
    total = risk_units.sum(axis=1).replace(0, np.nan)
    rc = risk_units.div(total, axis=0).fillna(0.0)
    return rc


risk_contrib_equal = diagonal_risk_contributions(investable_weights["equal_weight"], ewma_vol)
risk_contrib_risk_budget = diagonal_risk_contributions(investable_weights["risk_budget"], ewma_vol)
risk_contrib_port_vol = diagonal_risk_contributions(investable_weights["signal_portfolio_vol_target"], ewma_vol)

risk_contrib_summary = pd.DataFrame({
    "equal_weight_avg_rc": risk_contrib_equal.mean(),
    "risk_budget_avg_rc": risk_contrib_risk_budget.mean(),
    "signal_port_vol_avg_rc": risk_contrib_port_vol.mean(),
})

risk_contrib_summary

In [ ]:
plt.figure(figsize=(10, 6))
risk_contrib_summary.sort_values("risk_budget_avg_rc")["risk_budget_avg_rc"].plot(kind="barh")
plt.title("Average Diagonal Risk Contribution: Risk Budget Strategy")
plt.xlabel("Average risk contribution")
plt.ylabel("Asset")
plt.show()

## 23. Futures-style contract sizing example

For futures, target notional weight can be converted into contracts:

$$
contracts_{i,t} = \frac{NAV_t w_{i,t}} {P_{i,t} \times multiplier_i}
$$

This is a simplified example without margin, tick size, or exchange constraints.

In [ ]:
def simulate_prices_from_returns(returns, start_price=100.0):
    return start_price * (1 + returns).cumprod()


prices = pd.DataFrame(index=returns.index)
start_prices = {
    "GOLD": 1900.0,
    "OIL": 75.0,
}
for asset in asset_cols:
    prices[asset] = simulate_prices_from_returns(returns[asset], start_price=start_prices.get(asset, 100.0))


def futures_contract_sizing(weights, prices, nav, asset_metadata):
    futures_assets = asset_metadata.loc[asset_metadata["is_futures"], "asset"].tolist()
    multipliers = asset_metadata.set_index("asset")["contract_multiplier"]

    rows = []

    for asset in futures_assets:
        notional = nav * weights[asset]
        contract_value = prices[asset] * multipliers.loc[asset]
        contracts = notional / contract_value.replace(0, np.nan)

        rows.append(pd.DataFrame({
            "date": weights.index,
            "asset": asset,
            "target_weight": weights[asset],
            "nav": nav,
            "price": prices[asset],
            "contract_multiplier": multipliers.loc[asset],
            "target_contracts": contracts
        }))

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


contract_sizing = futures_contract_sizing(
    weights_signal_port_vol,
    prices,
    backtests["signal_portfolio_vol_target"]["nav"],
    asset_metadata
)

contract_sizing.head()

In [ ]:
contract_summary = (
    contract_sizing
    .groupby("asset")
    .agg(
        mean_abs_contracts=("target_contracts", lambda x: np.mean(np.abs(x))),
        p95_abs_contracts=("target_contracts", lambda x: np.quantile(np.abs(x), 0.95)),
        max_abs_contracts=("target_contracts", lambda x: np.max(np.abs(x)))
    )
    .reset_index()
)

contract_summary

## 24. Volatility targeting failure modes

Volatility targeting can fail when:

1. volatility jumps faster than the estimator reacts;
2. leverage is high before a shock;
3. all investors delever simultaneously;
4. transaction costs spike;
5. liquidity disappears;
6. correlations rise;
7. drawdown overlays sell after losses;
8. target volatility is too aggressive.

A good portfolio process treats vol targeting as one control layer, not the entire risk system.

## 25. Audit checks

We check:

1. volatility forecasts are shifted;
2. no infinite weights;
3. gross exposure cap respected;
4. transaction costs non-negative;
5. NAV remains positive;
6. target-vol strategy realised volatility is reported;
7. turnover is finite.

In [ ]:
def run_sizing_audit(backtests, strategies, ewma_vol, config):
    rows = []

    for name, weights in strategies.items():
        finite_weights = np.isfinite(weights.to_numpy()).all()
        gross_cap_ok = bool((weights.abs().sum(axis=1) <= config.max_portfolio_gross + 1e-8).all())
        costs_ok = bool((backtests[name]["transaction_cost"] >= -1e-12).all())
        nav_ok = bool((backtests[name]["nav"] > 0).all())
        turnover_finite = bool(np.isfinite(backtests[name]["turnover"]).all())

        realised_vol = float(backtests[name]["net_return"].std(ddof=1) * np.sqrt(config.annualisation))

        rows.append({
            "strategy": name,
            "finite_weights": finite_weights,
            "gross_cap_ok": gross_cap_ok,
            "costs_nonnegative": costs_ok,
            "nav_positive": nav_ok,
            "turnover_finite": turnover_finite,
            "realised_vol_ann": realised_vol,
            "passed": finite_weights and gross_cap_ok and costs_ok and nav_ok and turnover_finite
        })

    return pd.DataFrame(rows)


audit_report = run_sizing_audit(backtests, strategies, ewma_vol, config)
audit_report

## 26. Saving outputs

The notebook saves:

1. synthetic returns;
2. asset metadata;
3. volatility estimates;
4. signals;
5. all strategy weights;
6. backtest returns;
7. performance summaries;
8. rolling strategy volatility;
9. regime analysis;
10. cost summary;
11. risk contribution reports;
12. contract sizing examples;
13. audit report;
14. manifest.

In [ ]:
output_dir = Path("data/processed/volatility_targeting_and_position_sizing_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
metadata_path = output_dir / "asset_metadata.csv"
roll_vol_path = output_dir / "rolling_volatility.csv"
ewma_vol_path = output_dir / "ewma_volatility.csv"
signals_path = output_dir / "signals.csv"
perf_path = output_dir / "performance_summary.csv"
perf_overlay_path = output_dir / "performance_with_overlay.csv"
strategy_vol_path = output_dir / "strategy_rolling_vol.csv"
regime_analysis_path = output_dir / "regime_analysis.csv"
cost_summary_path = output_dir / "cost_summary.csv"
risk_contrib_summary_path = output_dir / "risk_contrib_summary.csv"
contract_sizing_path = output_dir / "contract_sizing.csv"
contract_summary_path = output_dir / "contract_summary.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
roll_vol.to_csv(roll_vol_path)
ewma_vol.to_csv(ewma_vol_path)
signals.to_csv(signals_path)
perf.to_csv(perf_path, index=False)
perf_with_overlay.to_csv(perf_overlay_path, index=False)
strategy_rolling_vol.to_csv(strategy_vol_path)
regime_analysis.to_csv(regime_analysis_path, index=False)
cost_summary.to_csv(cost_summary_path, index=False)
risk_contrib_summary.to_csv(risk_contrib_summary_path)
contract_sizing.to_csv(contract_sizing_path, index=False)
contract_summary.to_csv(contract_summary_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

# Save strategy-specific weights and backtests.
weights_paths = {}
backtest_paths = {}

for name, weights in strategies.items():
    path = output_dir / f"weights_{name}.csv"
    weights.to_csv(path)
    weights_paths[name] = str(path)

for name, bt in backtests.items():
    path = output_dir / f"backtest_{name}.csv"
    bt.to_csv(path)
    backtest_paths[name] = str(path)

manifest = {
    "dataset_name": "volatility_targeting_and_position_sizing_outputs",
    "schema_version": "volatility_targeting_and_position_sizing_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "strategies": list(backtests.keys()),
    "performance_summary": perf_with_overlay.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "audit_report": audit_report.to_dict(orient="records"),
    "weights_files": weights_paths,
    "backtest_files": backtest_paths,
    "notes": [
        "Volatility estimates are shifted by one day before use in sizing.",
        "Equal weight, inverse vol, risk budget, asset-level vol target, and portfolio-level vol target sizing are compared.",
        "Position weights are shifted by one day before applying asset returns.",
        "Transaction costs are proportional to turnover.",
        "Drawdown-aware overlay is included as a simple risk-control example.",
        "Futures contract sizing is illustrative and excludes margin, tick values, and exchange constraints."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, perf_overlay_path, audit_report_path, contract_summary_path, manifest_path

## 27. Limitations

### 27.1 Synthetic returns

The notebook uses synthetic returns.

Real data requires accurate total-return series, futures rolls, corporate actions, and exchange calendars.

### 27.2 Diagonal risk approximation

Some risk-contribution and portfolio-vol estimates use diagonal covariance.

Correlations matter, especially in stress.

### 27.3 Simple volatility estimators

Rolling and EWMA volatility are useful but limited.

More advanced models include GARCH, HAR-RV, stochastic volatility, and realised intraday estimators.

### 27.4 Simple cost model

Costs are fixed basis points.

Real costs depend on spread, liquidity, participation, volatility, venue, and market impact.

### 27.5 No margin or financing

Leverage is capped, but financing and margin are ignored.

### 27.6 Drawdown overlay is crude

Drawdown-aware deleveraging can reduce losses or miss recoveries.

It requires careful governance.

### 27.7 No full covariance targeting

This notebook prepares the ground for covariance-aware portfolio construction in later notebooks.

## 28. Research frontier and extensions

Important extensions include:

1. **Full covariance volatility targeting**  
   Use $\sqrt{w^\top \Sigma w}$ with shrinkage covariance.

2. **GARCH/HAR volatility forecasts**  
   Use better forecasts than rolling/EWMA.

3. **Volatility-of-volatility controls**  
   Penalise unstable vol forecasts.

4. **Dynamic leverage constraints**  
   Adjust leverage caps by liquidity and stress.

5. **Expected shortfall targeting**  
   Size positions by tail risk instead of volatility.

6. **Risk parity optimisation**  
   Equalise marginal risk contributions.

7. **Kelly and fractional Kelly sizing**  
   Use forecast return and variance, with strong shrinkage.

8. **Market-impact-aware sizing**  
   Limit positions by capacity and participation rate.

9. **Portfolio-level drawdown controls**  
   Combine trend, volatility, and drawdown overlays.

10. **Chinese futures sizing**  
   Incorporate contract multipliers, tick values, margin, price limits, and night-session volatility.

## 29. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_04_covariance_estimation_and_shrinkage**  
   Replace diagonal approximations with full covariance.

2. **04_05_mean_variance_optimization_constraints**  
   Use expected returns, risk, and constraints.

3. **04_06_risk_parity_and_equal_risk_contribution**  
   Formalise risk contribution allocation.

4. **04_07_var_cvar_and_stress_testing**  
   Tail-risk-based sizing.

5. **04_08_portfolio_drawdown_control**  
   More systematic drawdown overlays.

6. **05_03_transaction_costs_slippage_latency**  
   More realistic turnover and execution costs.

## 30. Summary

This notebook implemented volatility targeting and position sizing.

It showed how to:

1. simulate multi-asset returns with volatility regimes;
2. compute rolling and EWMA volatility forecasts;
3. shift volatility estimates to avoid look-ahead;
4. generate synthetic alpha signals;
5. build equal-weight, inverse-vol, risk-budget, asset-vol-targeted, and portfolio-vol-targeted weights;
6. cap leverage and gross exposure;
7. backtest weights with timing discipline;
8. include transaction costs;
9. compare realised volatility to target volatility;
10. inspect exposure, turnover, and cost drag;
11. add a drawdown-aware deleveraging overlay;
12. evaluate stress-regime behaviour;
13. approximate risk contributions;
14. convert futures weights into contract counts;
15. save outputs and audits.

The key computational takeaway:

> Position sizing is a timing-sensitive risk transformation from signals and volatility forecasts into weights.

The key financial takeaway:

> Volatility targeting can stabilise risk, but it can also increase leverage, turnover, and procyclical deleveraging if used blindly.

## 31. Further reading

- Moreira and Muir on volatility-managed portfolios.
- Grinold and Kahn, *Active Portfolio Management.*
- Meucci, *Risk and Asset Allocation.*
- Risk parity and volatility targeting literature.
- Futures CTA position sizing and volatility scaling research.